<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/06_Exercise_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise: Quantize and Serve a Model

In [ ]:
!pip install accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

## Clone llama.cpp (Version 2760) / Install GGUF-PY

In [ ]:
!git clone -b b2760 --single-branch https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp/gguf-py && pip install .

Cloning into 'llama.cpp'...
remote: Enumerating objects: 15623, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 15623 (delta 0), reused 1 (delta 0), pack-reused 15622 (from 1)
Receiving objects: 100% (15623/15623), 22.17 MiB | 8.92 MiB/s, done.
Resolving deltas: 100% (11009/11009), done.
Note: switching to '3f167476b11efa7ab08f6cacdeb8cab0935c1249'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Processing /content/llama.cpp/gguf-py
  Installing build dependencies ... done
  Getting requirements to build whee

## Supported Models

In [ ]:
import gguf
list(gguf.MODEL_ARCH)

[<MODEL_ARCH.LLAMA: 1>,
 <MODEL_ARCH.FALCON: 2>,
 <MODEL_ARCH.BAICHUAN: 3>,
 <MODEL_ARCH.GROK: 4>,
 <MODEL_ARCH.GPT2: 5>,
 <MODEL_ARCH.GPTJ: 6>,
 <MODEL_ARCH.GPTNEOX: 7>,
 <MODEL_ARCH.MPT: 8>,
 <MODEL_ARCH.STARCODER: 9>,
 <MODEL_ARCH.PERSIMMON: 10>,
 <MODEL_ARCH.REFACT: 11>,
 <MODEL_ARCH.BERT: 12>,
 <MODEL_ARCH.NOMIC_BERT: 13>,
 <MODEL_ARCH.BLOOM: 14>,
 <MODEL_ARCH.STABLELM: 15>,
 <MODEL_ARCH.QWEN: 16>,
 <MODEL_ARCH.QWEN2: 17>,
 <MODEL_ARCH.QWEN2MOE: 18>,
 <MODEL_ARCH.PHI2: 19>,
 <MODEL_ARCH.PHI3: 20>,
 <MODEL_ARCH.PLAMO: 21>,
 <MODEL_ARCH.CODESHELL: 22>,
 <MODEL_ARCH.ORION: 23>,
 <MODEL_ARCH.INTERNLM2: 24>,
 <MODEL_ARCH.MINICPM: 25>,
 <MODEL_ARCH.GEMMA: 26>,
 <MODEL_ARCH.STARCODER2: 27>,
 <MODEL_ARCH.MAMBA: 28>,
 <MODEL_ARCH.XVERSE: 29>,
 <MODEL_ARCH.COMMAND_R: 30>,
 <MODEL_ARCH.DBRX: 31>,
 <MODEL_ARCH.OLMO: 32>]

## Load Your Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Locutusque/gpt2-large-medical"
tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(name,
                                             torch_dtype=torch.float32,
                                             trust_remote_code=True,
                                             device_map={"": 0})

tokenizer_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/197 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [ ]:
model.__class__

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

## Save Model to Disk for Conversion

In [ ]:
!rm -rf ./model && mkdir model
model.save_pretrained('./model')
tokenizer.save_pretrained('./model')

('./model/tokenizer_config.json',
 './model/special_tokens_map.json',
 './model/vocab.json',
 './model/merges.txt',
 './model/added_tokens.json',
 './model/tokenizer.json')

## Convert Model to GGUF

In [ ]:
!python ./llama.cpp/convert-hf-to-gguf.py ./model

Loading model: model
gguf: This GGUF file is for Little Endian only
Set model parameters
Set model tokenizer
gguf: Adding 50000 merge(s).
gguf: Setting special token type bos to 50256
gguf: Setting special token type eos to 50256
gguf: Setting special token type unk to 50256
gguf: Setting special token type pad to 50257
Exporting model to 'model/ggml-model-f16.gguf'
gguf: loading model part 'model.safetensors'
blk.0.attn_qkv.bias, n_dims = 1, torch.float32 --> float32
blk.0.attn_qkv.weight, n_dims = 2, torch.float32 --> float16
blk.0.attn_output.bias, n_dims = 1, torch.float32 --> float32
blk.0.attn_output.weight, n_dims = 2, torch.float32 --> float16
blk.0.attn_norm.bias, n_dims = 1, torch.float32 --> float32
blk.0.attn_norm.weight, n_dims = 1, torch.float32 --> float32
blk.0.ffn_norm.bias, n_dims = 1, torch.float32 --> float32
blk.0.ffn_norm.weight, n_dims = 1, torch.float32 --> float32
blk.0.ffn_up.bias, n_dims = 1, torch.float32 --> float32
blk.0.ffn_up.weight, n_dims = 2, torch.fl

## Builds llama.cpp for Quantization

In [ ]:
!cd llama.cpp && make clean && make
!pip install -r llama.cpp/requirements.txt

I ccache not found. Consider installing it for faster compilation.
I llama.cpp build info: 
I UNAME_S:   Linux
I UNAME_P:   x86_64
I UNAME_M:   x86_64
I CFLAGS:    -I. -Icommon -D_XOPEN_SOURCE=600 -D_GNU_SOURCE -DNDEBUG -DGGML_USE_LLAMAFILE  -std=c11   -fPIC -O3 -Wall -Wextra -Wpedantic -Wcast-qual -Wno-unused-function -Wshadow -Wstrict-prototypes -Wpointer-arith -Wmissing-prototypes -Werror=implicit-int -Werror=implicit-function-declaration -pthread -march=native -mtune=native -Wdouble-promotion 
I CXXFLAGS:  -std=c++11 -fPIC -O3 -Wall -Wextra -Wpedantic -Wcast-qual -Wno-unused-function -Wmissing-declarations -Wmissing-noreturn -pthread  -march=native -mtune=native -Wno-array-bounds -Wno-format-truncation -Wextra-semi -I. -Icommon -D_XOPEN_SOURCE=600 -D_GNU_SOURCE -DNDEBUG -DGGML_USE_LLAMAFILE 
I NVCCFLAGS: -std=c++11 -O3 
I LDFLAGS:    
I CC:        cc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
I CXX:       c++ (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0

rm -vrf *.o tests/*.o *.so *.a *.dll be

In [ ]:
!pip install numpy==1.26.4 --force-reinstall

## Quantized GGUF Model

In [ ]:
!./llama.cpp/quantize ./model/ggml-model-f16.gguf ./model/ggml-model-q8_0.gguf q8_0

main: build = 2760 (3f16747)
main: built with cc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0 for x86_64-linux-gnu
main: quantizing './model/ggml-model-f16.gguf' to './model/ggml-model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 17 key-value pairs and 437 tensors from ./model/ggml-model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gpt2
llama_model_loader: - kv   1:                               general.name str              = model
llama_model_loader: - kv   2:                           gpt2.block_count u32              = 36
llama_model_loader: - kv   3:                        gpt2.context_length u32              = 1024
llama_model_loader: - kv   4:                      gpt2.embedding_length u32              = 1280
llama_model_loader: - kv   5:                   gpt2.feed_forward_length u32        

## Install Ollama

In [ ]:
!curl https://ollama.ai/install.sh | sh
!pip install ollama

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  33910      0 --:--:-- --:--:-- --:--:-- 33966
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Helper Functions for Serving Ollama

In [ ]:
# https://stackoverflow.com/questions/77697302/how-to-run-ollama-in-google-colab

import os
import asyncio

# NB: You may need to set these depending and get cuda working depending which backend you are running.
# Set environment variable for NVIDIA library
# Set environment variables for CUDA
os.environ['PATH'] += ':/usr/local/cuda/bin'
# Set LD_LIBRARY_PATH to include both /usr/lib64-nvidia and CUDA lib directories
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64'

async def run_process(cmd):
    print('>>> starting', *cmd)
    process = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )

    # define an async pipe function
    async def pipe(lines):
        async for line in lines:
            print(line.decode().strip())

        await asyncio.gather(
            pipe(process.stdout),
            pipe(process.stderr),
        )

    # call it
    await asyncio.gather(pipe(process.stdout), pipe(process.stderr))

In [ ]:
import asyncio
import threading

async def start_ollama_serve():
    await run_process(['ollama', 'serve'])

def run_async_in_thread(loop, coro):
    asyncio.set_event_loop(loop)
    loop.run_until_complete(coro)
    loop.close()

# Create a new event loop that will run in a new thread
new_loop = asyncio.new_event_loop()

# Start ollama serve in a separate thread so the cell won't block execution
thread = threading.Thread(target=run_async_in_thread, args=(new_loop, start_ollama_serve()))
thread.start()

>>> starting ollama serve


## Custom Model File

In [ ]:
from transformers import AutoTokenizer
name = "Locutusque/gpt2-large-medical"
tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)

In [ ]:
print(tokenizer.chat_template)

None


In [ ]:
tokenizer.all_special_tokens, tokenizer.additional_special_tokens

(['<|endoftext|>', '[PAD]', '<|USER|>', '<|ASSISTANT|>'],
 ['<|USER|>', '<|ASSISTANT|>'])

In [ ]:
# https://huggingface.co/Locutusque/gpt2-large-conversational

modelfile = ('FROM ./model/ggml-model-q8_0.gguf'
'\n\nTEMPLATE """'
'\n<|USER|> {{ .Prompt }} <|ASSISTANT|> {{ .Response }} <|endoftext|>'
'\n"""'
'\n\nPARAMETER stop <|endoftext|>'
'\nPARAMETER stop [PAD]'
'\nPARAMETER stop <|ASSISTANT|>')

In [ ]:
print(modelfile)

with open('modelfile_custom', 'w') as f:
    f.write(modelfile)

FROM ./model/ggml-model-q8_0.gguf

TEMPLATE """
<|USER|> {{ .Prompt }} <|ASSISTANT|> {{ .Response }} <|endoftext|>
"""

PARAMETER stop <|endoftext|>
PARAMETER stop [PAD]
PARAMETER stop <|ASSISTANT|>


## Loading Model into Ollama

In [ ]:
!ollama create -f modelfile_custom medical

[GIN] 2025/05/20 - 14:29:10 | 200 |      80.639µs |       127.0.0.1 | HEAD     "/"
[GIN] 2025/05/20 - 14:29:24 | 201 |  8.400925942s |       127.0.0.1 | POST     "/api/blobs/sha256:6829c9d762ab411fa8805c678d707981c220b35f227be522a643dc4123bbd738"
[GIN] 2025/05/20 - 14:29:24 | 200 |     9.89051ms |       127.0.0.1 | POST     "/api/create"



In [ ]:
!ollama list

[GIN] 2025/05/20 - 14:29:24 | 200 |      35.032µs |       127.0.0.1 | HEAD     "/"
[GIN] 2025/05/20 - 14:29:24 | 200 |     482.965µs |       127.0.0.1 | GET      "/api/tags"
NAME              ID              SIZE      MODIFIED               
medical:latest    031eda12f087    895 MB    Less than a second ago    


## Querying the Model

In [ ]:
import ollama
response = ollama.chat(model='medical', messages=[
  {
    'role': 'user',
    'content': 'I got a bad headache on the left side of my forehead. How can I get better?',
  },
], stream=False)

[GIN] 2025/05/20 - 14:56:00 | 200 |  366.109423ms |       127.0.0.1 | POST     "/api/chat"


In [ ]:
print(response['message']['content'])

 an over-the-counter medicine called ibuprofen may help relieve your headache, according to the American Academy of Neurology. 
